# 8. Streaming

Instead of waiting for a model to finish generating a full response, a chain can
stream its output chunk by chunk as it's produced via `.astream(...)`. Jupyter
supports top-level `await`, so this works directly in a notebook cell.

**Prerequisites:** Ollama running locally with `llama3.2` pulled.

### Setup

This cell makes the project's shared `tools`/`models` packages importable
regardless of where Jupyter's working directory actually is (it's usually
this notebook's own folder, not the repo root), and loads `.env` plus any
cached secrets in `.env.local` (populated by `scripts/lib/env.sh` the first
time you've run `scripts/start_app.sh` / `scripts/start_infra.sh`).

In [ ]:
import sys
from pathlib import Path

from dotenv import load_dotenv

project_root = Path.cwd()
while not (project_root / "pyproject.toml").exists():
    project_root = project_root.parent
sys.path.insert(0, str(project_root))

load_dotenv(project_root / ".env")
load_dotenv(project_root / ".env.local", override=True)  # cached secrets, if resolve_env() has run at least once
print("Project root on sys.path:", project_root)

In [ ]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate

from models.chat_models.ollama_models import SupportedModel, get_chat_model

llm = get_chat_model(SupportedModel.llama3_2)
prompt = ChatPromptTemplate.from_messages([("human", "Write a short, engaging explanation of: {topic}")])
chain = prompt | llm | StrOutputParser()

In [ ]:
async for chunk in chain.astream({"topic": "how rainbows form"}):
    print(chunk, end="", flush=True)

## 🧪 Playground

**1. Count the chunks** instead of printing them, to see how many pieces the response actually arrives in.

In [ ]:
# TODO: accumulate chunks into a list via async for, then print len(chunks)


**2. Try `gemma4`** — a documented quirk: `ChatOllama.astream()` interleaves real chunks with *empty*-string chunks for this model (verified: the real text is all still there if you don't filter blanks). Confirm this live.

In [ ]:
# TODO: llm_gemma = get_chat_model(SupportedModel.gemma4); rebuild the chain and stream, printing repr(chunk) to see empties


**3. A longer topic** — stream an explanation of something more involved and watch the pacing.

In [ ]:
# TODO: stream a longer/more complex topic
